# 마무리요약 텍스트 골드셋

각 강의의 **마지막 30분** 전사 텍스트를 LLM에 전달해 **마무리요약 텍스트**를 강의당 1개 추출한다.

추출 우선순위:
1. 강사가 오늘 배운 핵심 내용을 **요약·정리**하는 구간
2. 없으면 강의 마무리 인사·공지 등 **마무리 멘트** 구간

흐름:
1. CSV 로드 → 날짜별 마지막 30분 문장 추출
2. LLM으로 마무리요약 스팬(`start_index`~`end_index`) 추출
3. `*_closing_goldset_v2.json/xlsx` 저장
4. 사람 검수 후 확정 → `*_closing_goldset_final.json`

In [13]:
import json
import re
from datetime import timedelta
from pathlib import Path

import openpyxl
import pandas as pd
import google.generativeai as genai
from openpyxl.styles import Alignment
from tqdm.auto import tqdm

import sys
sys.path.insert(0, str(Path(".").resolve().parent))
from app.core.config import settings

genai.configure(api_key=settings.api_key)
MODEL    = settings.eval_model
GS       = Path("../data/goldset")
CSV_PATH = Path("../data/processed/lectures_kss.csv")

LAST_MIN = 30  # 마지막 몇 분을 입력으로 줄지

df_all = pd.read_csv(CSV_PATH)
df_all["ts"] = pd.to_timedelta(df_all["timestamp"])

DATES = [f"2026-02-{d:02d}" for d in range(9, 14)]
print(f"대상 날짜 {len(DATES)}개  |  model: {MODEL}")
for date in DATES:
    d = df_all[df_all["date"] == date]
    max_ts  = d["ts"].max()
    cutoff  = max_ts - timedelta(minutes=LAST_MIN)
    last30  = d[d["ts"] >= cutoff]
    print(f"  {date}: 전체 {len(d):4d}문장  →  마지막 {LAST_MIN}분 {len(last30):3d}문장 "
          f"(전역 {last30.index[0]}~{last30.index[-1]})")

대상 날짜 5개  |  model: models/gemini-2.5-pro
  2026-02-09: 전체 1646문장  →  마지막 30분 182문장 (전역 9984~10165)
  2026-02-10: 전체 1815문장  →  마지막 30분 182문장 (전역 11643~11824)
  2026-02-11: 전체 1660문장  →  마지막 30분 167문장 (전역 13482~13648)
  2026-02-12: 전체 1037문장  →  마지막 30분 160문장 (전역 15178~15337)
  2026-02-13: 전체 1543문장  →  마지막 30분 160문장 (전역 16067~16226)


---

## 프롬프트 설정

**마무리요약** 정의를 아래 프롬프트에서 직접 수정하세요.

In [14]:
CLS_SYSTEM = """당신은 강의 전사 텍스트에서 '마무리요약' 구간을 식별하는 전문가입니다.
응답은 반드시 JSON만 출력하세요."""

CLS_USER = """아래는 강의 마지막 30분의 전사 텍스트입니다. 각 줄은 [전역인덱스] 문장 형식입니다.

이 구간에서 **마무리요약** 텍스트를 하나만 식별하세요.

【마무리요약 정의 — 우선순위 순】
① (1순위) 강사가 오늘 배운 핵심 내용을 요약·정리하는 구간
   - "오늘 배운 것은...", "정리하면...", "핵심은..." 등 명시적 요약
   - 오늘 다룬 개념·실습·쿼리를 나열·복습하는 구간
② (2순위) 강사가 수업을 마무리하는 멘트 구간 (1순위가 없을 때만)
   - "수고하셨습니다", "다음 시간에...", 과제 안내, 쉬는시간 공지 등

【반드시 제외】
- 단순 실습·예제 진행 중인 구간
- 질의응답만 있는 구간 (마무리 요약 없이)
- 강의 시작 인사, 오늘 일정 소개

식별된 마무리요약 구간에 대해:
- closing_type: "summary"(핵심요약) 또는 "closing_remark"(마무리멘트)
- start_index / end_index: 시작/끝 문장의 전역 인덱스 (반드시 아래 구간 내 값)
- key_sentence: 마무리 의도를 가장 잘 드러내는 핵심 문장 원문 1개
- closing_text: start_index~end_index 범위의 문장을 공백으로 이어붙인 전체 텍스트

마무리 구간이 전혀 없으면 {{"closing": null}} 로 응답하세요.

문장 구간 (마지막 {last_min}분):
{indexed_sentences}

아래 JSON 형식으로만 응답하세요:
{{"closing": {{"closing_type": "summary", "start_index": 0, "end_index": 0, "key_sentence": "...", "closing_text": "..."}}}}"""


def _format_span(sentences, indices):
    return "\n".join(f"[{i}] {sentences[i]}" for i in indices)


def _parse_closing(raw):
    text = re.sub(r"^```\w*\s*", "", raw.strip())
    text = re.sub(r"\s*```$", "", text).strip()
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if not m:
            return None
        data = json.loads(m.group())
    return data.get("closing")

In [15]:
def run_closing_extraction(date, df_date):
    df_date = df_date.reset_index(drop=True)
    sentences = df_date["text_raw"].fillna("").tolist()
    df_date["ts"] = pd.to_timedelta(df_date["timestamp"])

    # 타임스탬프 역주행 지점 탐지 → 마지막 세션(오후)만 사용
    backward_mask = df_date["ts"].diff() < pd.Timedelta(0)
    session_starts = df_date[backward_mask].index.tolist()
    if session_starts:
        last_session_start = session_starts[-1]
        df_session = df_date.iloc[last_session_start:]
        print(f"  오전/오후 분리: 오후 세션 row {last_session_start}~ 사용 "
              f"({df_session.iloc[0]['timestamp']} ~ {df_session.iloc[-1]['timestamp']})")
    else:
        df_session = df_date

    max_ts = df_session["ts"].max()
    cutoff = max_ts - timedelta(minutes=LAST_MIN)
    last30_idx = df_session[df_session["ts"] >= cutoff].index.tolist()  # 로컬 인덱스

    # 전역 인덱스 변환 (int() 캐스팅으로 numpy.int64 → Python int)
    global_offset = int(df_all[df_all["date"] == date].index[0])
    global_indices = [i + global_offset for i in last30_idx]

    indexed_text = "\n".join(
        f"[{gi}] {sentences[li]}" for li, gi in zip(last30_idx, global_indices)
    )

    prompt = (
        f"{CLS_SYSTEM}\n\n"
        + CLS_USER.format(
            last_min=LAST_MIN,
            indexed_sentences=indexed_text,
        )
    )

    model = genai.GenerativeModel(MODEL)
    resp  = model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(
            response_mime_type="application/json",
            temperature=settings.llm_temperature,
        ),
    )
    closing = _parse_closing(resp.text)

    if not closing:
        return None, global_indices[0], global_indices[-1]

    try:
        gs, ge = int(closing["start_index"]), int(closing["end_index"])
    except (KeyError, ValueError, TypeError):
        return None, global_indices[0], global_indices[-1]

    if gs > ge:
        gs, ge = ge, gs
    gs = max(gs, global_indices[0])
    ge = min(ge, global_indices[-1])

    ls = gs - global_offset
    le = ge - global_offset
    ls = max(ls, 0)
    le = min(le, len(sentences) - 1)

    result = {
        "closing_type":     str(closing.get("closing_type", "")).strip(),
        "start":            gs,
        "end":              ge,
        "timestamp_start":  df_date.loc[ls, "timestamp"],
        "timestamp_end":    df_date.loc[le, "timestamp"],
        "key_sentence":     str(closing.get("key_sentence", "")).strip(),
        "closing_text":     " ".join(sentences[i] for i in range(ls, le + 1)),
    }
    return result, global_indices[0], global_indices[-1]

In [16]:
def save_draft(date, closing_raw, input_start, input_end):
    draft = {
        "source":       CSV_PATH.name,
        "date":         date,
        "method":       "llm_extraction_last_30min",
        "input_span":   {"start": input_start, "end": input_end},
        "closing":      closing_raw,
    }
    json_path = GS / f"{date}_closing_goldset_v2.json"
    json_path.write_text(json.dumps(draft, ensure_ascii=False, indent=2), encoding="utf-8")

    if closing_raw:
        row = {
            "date":            date,
            "closing_type":    closing_raw["closing_type"],
            "span":            f"{closing_raw['start']}~{closing_raw['end']}",
            "timestamp_start": closing_raw["timestamp_start"],
            "timestamp_end":   closing_raw["timestamp_end"],
            "key_sentence":    closing_raw["key_sentence"],
            "closing_text":    closing_raw["closing_text"],
            "human_check":     "",
        }
    else:
        row = {
            "date":            date,
            "closing_type":    "(없음)",
            "span":            "",
            "timestamp_start": "",
            "timestamp_end":   "",
            "key_sentence":    "",
            "closing_text":    "",
            "human_check":     "",
        }

    xlsx_path = GS / f"{date}_closing_goldset_v2.xlsx"
    pd.DataFrame([row]).to_excel(xlsx_path, index=False)

    wb = openpyxl.load_workbook(xlsx_path)
    ws = wb.active
    col_widths = {"A": 13, "B": 18, "C": 14, "D": 13, "E": 13, "F": 60, "G": 80, "H": 12}
    for col, w in col_widths.items():
        ws.column_dimensions[col].width = w
    for row_cells in ws.iter_rows(min_row=2):
        for cell in row_cells:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
    wb.save(xlsx_path)

    label = closing_raw["closing_type"] if closing_raw else "(없음)"
    print(f"  저장: {json_path.name}  →  {label}")
    return json_path, xlsx_path


# ── 전체 날짜 일괄 처리 ───────────────────────────────────────────────
all_results = {}
for date in tqdm(DATES, desc="closing-extract"):
    df_date = df_all[df_all["date"] == date].copy()
    if df_date.empty:
        print(f"[{date}] 데이터 없음 → 건너뜀")
        continue

    print(f"\n[{date}]")
    closing_raw, inp_s, inp_e = run_closing_extraction(date, df_date)

    if closing_raw:
        print(f"  [{closing_raw['start']}~{closing_raw['end']}] "
              f"{closing_raw['closing_type']}  "
              f"{closing_raw['timestamp_start']}~{closing_raw['timestamp_end']}")
        print(f"  핵심문장: {closing_raw['key_sentence'][:60]}")
    else:
        print("  마무리 구간 없음")

    all_results[date] = closing_raw
    save_draft(date, closing_raw, inp_s, inp_e)

print("\n완료. 날짜별 *_closing_goldset_v2.json/xlsx 저장됨.")
print("검수 표기: O=맞음 / X=아님 / M=범위수정필요")

closing-extract:   0%|          | 0/5 [00:00<?, ?it/s]


[2026-02-09]
  오전/오후 분리: 오후 세션 row 828~ 사용 (01:17:10 ~ 04:10:46)


closing-extract:  20%|██        | 1/5 [00:30<02:00, 30.11s/it]

  [10974~10976] summary  04:09:59~04:10:10
  핵심문장: 그럼 여기서 펑션까지는 다 일단은 펑션은 여러분 암기하는 게 아니라 할 때마다 찾아보는 느낌이에요.
  저장: 2026-02-09_closing_goldset_v2.json  →  summary

[2026-02-10]
  오전/오후 분리: 오후 세션 row 841~ 사용 (01:11:49 ~ 05:50:53)


closing-extract:  40%|████      | 2/5 [00:43<01:00, 20.10s/it]

...

  [14457~14458] closing_remark  05:50:20~05:50:32
  핵심문장: 여러분 오늘 많으셨습니다.
  저장: 2026-02-11_closing_goldset_v2.json  →  closing_remark

[2026-02-12]
  오전/오후 분리: 오후 세션 row 879~ 사용 (01:13:20 ~ 01:38:18)


closing-extract:  80%|████████  | 4/5 [01:16<00:18, 18.62s/it]

  [15491~15492] summary  01:37:22~01:37:47
  핵심문장: 인포메이션 스키마는 각각의 정보를 가지고 있는 테이블 단위로 되어 있고 거기 안에 필드가 여러 개가 있는데,
  저장: 2026-02-12_closing_goldset_v2.json  →  summary

[2026-02-13]
  오전/오후 분리: 오후 세션 row 731~ 사용 (01:11:16 ~ 05:50:25)


closing-extract: 100%|██████████| 5/5 [01:25<00:00, 17.13s/it]

  [17034~17038] closing_remark  05:50:05~05:50:25
  핵심문장: 여러분 한 주간 고생 많으셨습니다.
  저장: 2026-02-13_closing_goldset_v2.json  →  closing_remark

완료. 날짜별 *_closing_goldset_v2.json/xlsx 저장됨.
검수 표기: O=맞음 / X=아님 / M=범위수정필요


---

## 확정 — 검수 결과 반영

각 날짜의 `*_closing_goldset_v2.xlsx`에서 `human_check` 열을 채운 뒤 아래 셀을 실행해 확정본을 저장한다.
- `OVERRIDE_BY_DATE`: 날짜별로 span 범위를 직접 지정할 때 사용
- `human_check == 'O'`인 항목만 확정본에 포함

In [ ]:
# 필요 시 날짜별 span 수동 지정 (start, end 는 전역 인덱스)
OVERRIDE_BY_DATE = {
    "2026-02-09": {"start": 10962, "end": 10983},  # 식별/비식별 요약 + 펑션 정리 + 마무리 인사 포함
}

for date in DATES:
    xlsx_path = GS / f"{date}_closing_goldset_v2.xlsx"
    json_path = GS / f"{date}_closing_goldset_v2.json"
    if not xlsx_path.exists() or not json_path.exists():
        print(f"[{date}] v2 파일 없음 → 건너뜀")
        continue

    review = pd.read_excel(xlsx_path)
    v2     = json.loads(json_path.read_text(encoding="utf-8"))

    hc = str(review.iloc[0]["human_check"]).strip().upper() if not review.empty else ""
    if hc != "O":
        print(f"[{date}] human_check='{hc}' → 건너뜀 (O 아님)")
        continue

    closing = v2.get("closing")
    if not closing:
        print(f"[{date}] closing 없음 → 건너뜀")
        continue

    # 수동 override 적용
    if date in OVERRIDE_BY_DATE:
        ov = OVERRIDE_BY_DATE[date]
        df_date = df_all[df_all["date"] == date].reset_index(drop=True)
        global_offset = int(df_all[df_all["date"] == date].index[0])
        sentences = df_date["text_raw"].fillna("").tolist()
        ls = ov["start"] - global_offset
        le = ov["end"]   - global_offset
        closing["start"]           = ov["start"]
        closing["end"]             = ov["end"]
        closing["timestamp_start"] = df_date.loc[ls, "timestamp"]
        closing["timestamp_end"]   = df_date.loc[le, "timestamp"]
        closing["closing_text"]    = " ".join(sentences[i] for i in range(ls, le + 1))
        print(f"[{date}] override 적용: {ov['start']}~{ov['end']}")

    final_obj = {
        "source":     v2["source"],
        "date":       date,
        "method":     "human_reviewed",
        "input_span": v2["input_span"],
        "closing":    closing,
    }
    final_path = GS / f"{date}_closing_goldset_final.json"
    final_path.write_text(json.dumps(final_obj, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"[{date}] 확정 → {final_path.name}")
    print(f"  [{closing['start']}~{closing['end']}] {closing['closing_type']} "
          f"{closing['timestamp_start']}~{closing['timestamp_end']}")
    print(f"  핵심문장: {closing['key_sentence'][:60]}")

print("\n완료.")